In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir('/content/drive/MyDrive/credit-risk-assessment-system')

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

X_train = pd.read_csv('data/processed/X_train_fe.csv')
X_test = pd.read_csv('data/processed/X_test_fe.csv')
y_train = pd.read_csv('data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

leftover_cols = [c for c in ['SK_ID_CURR', 'AMT_ANNUITY_RAW', 'AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW']
                  if c in X_train.columns]
X_train_model = X_train.drop(columns=leftover_cols)
X_test_model = X_test.drop(columns=leftover_cols)

print(X_train_model.shape, X_test_model.shape)

(246008, 191) (61503, 191)


In [3]:
!pip install optuna catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 16.8 MB/s eta 0:00:00


In [4]:
print("Total raw columns loaded:", X_train.shape[1])
print("Leftover columns found and dropped:", leftover_cols)

Total raw columns loaded: 195
Leftover columns found and dropped: ['SK_ID_CURR', 'AMT_ANNUITY_RAW', 'AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW']


In [5]:
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_state': 42,
        'verbose': 0
    }

    model = CatBoostClassifier(**params)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_model, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)

    return scores.mean()

In [6]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("Best AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-08-08 11:09:52,431] A new study created in memory with name: no-name-c538a603-afc8-485e-a2f1-23aea403b768
[I 2026-08-08 11:10:52,903] Trial 0 finished with value: 0.7567103037063682 and parameters: {'iterations': 186, 'depth': 7, 'learning_rate': 0.10933887278428653, 'l2_leaf_reg': 8.373346737785317}. Best is trial 0 with value: 0.7567103037063682.
[I 2026-08-08 11:12:14,627] Trial 1 finished with value: 0.7461659790351088 and parameters: {'iterations': 230, 'depth': 8, 'learning_rate': 0.01309532366792909, 'l2_leaf_reg': 6.083325102747847}. Best is trial 0 with value: 0.7567103037063682.
[I 2026-08-08 11:13:47,716] Trial 2 finished with value: 0.7511389675634014 and parameters: {'iterations': 458, 'depth': 5, 'learning_rate': 0.28538658996976923, 'l2_leaf_reg': 7.9373202939185195}. Best is trial 0 with value: 0.7567103037063682.
[I 2026-08-08 11:25:25,229] Trial 6 finished with value: 0.7557264452279483 and parameters: {'iterations': 353, 'depth': 10, 'learning_rate': 0.032256

Best AUC: 0.7578568314015679
Best params: {'iterations': 477, 'depth': 7, 'learning_rate': 0.042106168367021204, 'l2_leaf_reg': 7.2626141545771805}


In [7]:
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbose'] = 0

final_catboost = CatBoostClassifier(**best_params)

skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
final_cv_scores = cross_val_score(final_catboost, X_train_model, y_train, cv=skf5, scoring='roc_auc', n_jobs=-1)

print("Tuned CatBoost 5-fold CV scores:", final_cv_scores)
print(f"Mean: {final_cv_scores.mean():.4f}, Std: {final_cv_scores.std():.4f}")

Tuned CatBoost 5-fold CV scores: [0.75607167 0.75693538 0.7605483  0.75950999 0.75822197]
Mean: 0.7583, Std: 0.0016


In [8]:
final_catboost.fit(X_train_model, y_train)

y_pred_proba_final = final_catboost.predict_proba(X_test_model)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba_final)
print(f"Final tuned CatBoost - Test AUC: {test_auc:.4f}")

Final tuned CatBoost - Test AUC: 0.7620


In [9]:
import joblib
joblib.dump(final_catboost, 'models/catboost_tuned_final.pkl')
print("Final model saved")

Final model saved
